In [1]:
# 1. IMPORT LIBRARIES
import os
import pandas as pd
import joblib

# preprocessing tools
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

# 2. LOAD TRAIN / TEST DATA
# load from processed folder (output of Task 1)
X_train = pd.read_csv("/content/X_train.csv")
X_test  = pd.read_csv("/content/X_test.csv")

print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)

X_train shape: (4504, 18)
X_test shape : (1126, 18)


In [2]:
# 3. IDENTIFY COLUMN TYPES
# numeric columns
num_cols = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()

# categorical columns
cat_cols = X_train.select_dtypes(include=["object"]).columns.tolist()

print("Numeric columns:", num_cols)
print("Categorical columns:", cat_cols)

Numeric columns: ['Tenure', 'CityTier', 'WarehouseToHome', 'HourSpendOnApp', 'NumberOfDeviceRegistered', 'SatisfactionScore', 'NumberOfAddress', 'Complain', 'OrderAmountHikeFromlastYear', 'CouponUsed', 'OrderCount', 'DaySinceLastOrder', 'CashbackAmount']
Categorical columns: ['PreferredLoginDevice', 'PreferredPaymentMode', 'Gender', 'PreferedOrderCat', 'MaritalStatus']


In [3]:
# 4. CREATE IMPUTER PIPELINE
# numeric imputer -> fill missing with median
num_imputer = SimpleImputer(strategy="median")

# categorical imputer -> fill missing with most frequent value (mode)
cat_imputer = SimpleImputer(strategy="most_frequent")

# combine both using ColumnTransformer
imputer_pipeline = ColumnTransformer(
    transformers=[
        ("num", num_imputer, num_cols),
        ("cat", cat_imputer, cat_cols)
    ]
)

In [4]:
# 5. FIT ONLY ON TRAIN (NO LEAKAGE)
# IMPORTANT:
# We only fit on X_train to avoid data leakage
imputer_pipeline.fit(X_train)

ColumnTransformer(transformers=[('num', SimpleImputer(strategy='median'),
                                 ['Tenure', 'CityTier', 'WarehouseToHome',
                                  'HourSpendOnApp', 'NumberOfDeviceRegistered',
                                  'SatisfactionScore', 'NumberOfAddress',
                                  'Complain', 'OrderAmountHikeFromlastYear',
                                  'CouponUsed', 'OrderCount',
                                  'DaySinceLastOrder', 'CashbackAmount']),
                                ('cat', SimpleImputer(strategy='most_frequent'),
                                 ['PreferredLoginDevice',
                                  'PreferredPaymentMode', 'Gender',
                                  'PreferedOrderCat', 'MaritalStatus'])])

In [5]:
# 6. TRANSFORM TRAIN & TEST
# apply transformation
X_train_clean = imputer_pipeline.transform(X_train)
X_test_clean  = imputer_pipeline.transform(X_test)

# convert back to DataFrame (important for later steps)
# ColumnTransformer changes order -> we must reconstruct columns
all_cols = num_cols + cat_cols

X_train_clean = pd.DataFrame(X_train_clean, columns=all_cols)
X_test_clean  = pd.DataFrame(X_test_clean, columns=all_cols)

print("After preprocessing:")
print("X_train_clean:", X_train_clean.shape)
print("X_test_clean :", X_test_clean.shape)

After preprocessing:
X_train_clean: (4504, 18)
X_test_clean : (1126, 18)


In [6]:
# 7. SAVE PIPELINE
joblib.dump(imputer_pipeline, "/content/imputer_pipeline.pkl")

print("Saved!")

# 8. SAVE CLEAN DATA
X_train_clean.to_csv("/content/X_train_clean.csv", index=False)
X_test_clean.to_csv("/content/X_test_clean.csv", index=False)

print("Saved!")

Saved!
Saved!
